In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\hp\Desktop\Amazon\CSV Files\amazon_india_2021.csv")

In [3]:
df.shape

(143715, 34)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138187 entries, 0 to 138186
Data columns (total 34 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   transaction_id          138187 non-null  object 
 1   order_date              138187 non-null  object 
 2   customer_id             138187 non-null  object 
 3   product_id              138187 non-null  object 
 4   product_name            138187 non-null  object 
 5   category                138187 non-null  object 
 6   subcategory             138187 non-null  object 
 7   brand                   138187 non-null  object 
 8   original_price_inr      138187 non-null  object 
 9   discount_percent        138187 non-null  float64
 10  discounted_price_inr    138187 non-null  float64
 11  quantity                138187 non-null  int64  
 12  subtotal_inr            138187 non-null  float64
 13  delivery_charges        127147 non-null  float64
 14  final_amount_inr    

In [4]:
import numpy as np

df.replace("", np.nan, inplace=True)


In [5]:
dfc = df.copy()

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [6]:
import pandas as pd

dfc['order_date'] = (
    dfc['order_date']
    .astype('string')
    .str.strip()
    .str.replace(r'\s+', '', regex=True)
)

dfc['order_date'] = pd.to_datetime(
    dfc['order_date'],
    dayfirst=True,
    errors='coerce'
)
dfc['order_date'] = dfc['order_date'].dt.strftime('%Y-%m-%d')


C:\Users\hp\AppData\Local\Temp\ipykernel_15856\570715499.py:10: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dfc['order_date'] = pd.to_datetime(


In [7]:
dfc['order_date'].head(20)

0     2021-01-29
1     2021-01-10
2     2021-01-18
3     2021-01-08
4     2021-01-02
5     2021-01-16
6     2021-01-09
7     2021-01-27
8     2021-01-30
9     2021-01-20
10    2021-01-07
11    2021-01-21
12    2021-01-14
13    2021-01-06
14    2021-01-04
15    2021-01-25
16    2021-01-23
17    2021-01-26
18    2021-01-21
19    2021-01-16
Name: order_date, dtype: object

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees.

In [8]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
        .astype(str)                      
        .str.replace('₹', '', regex=False) 
        .str.replace(',', '', regex=False)
        .str.replace('Rs ', '', regex=False)
        .str.strip()                
)

dfc['original_price_inr'] = pd.to_numeric(
    dfc['original_price_inr']
)


Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.

In [9]:
import re

def parse_rating(r):
    if pd.isna(r):
        return np.nan

    if '/' in r:
        a, b = r.split('/')
        return float(a) / float(b) * 5

    m = re.search(r'\d+\.?\d*', r)
    return float(m.group()) if m else np.nan


dfc['customer_rating'] = dfc['customer_rating'].apply(parse_rating)

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.

In [11]:
dfc['customer_city'] = (
    dfc['customer_city']
    .str.lower()
    .str.strip()
)
 
city_map = {
    'bangalore': 'Bengaluru',
    'bengaluru': 'Bengaluru',
    'bangalore/bengaluru': 'Bengaluru',
    'bengalore' : 'Bengaluru',
    'Bengaluru' : 'banglore',
    
    'mumbai': 'Mumbai',
    'bombay': 'Mumbai',
    'mumbai/bombay': 'Mumbai',
    'mumba' : 'Mumbai',
    'calcutta' : 'kolkata',

    'delhi': 'Delhi',
    'new delhi': 'Delhi',
    'delhi/new delhi': 'Delhi',
    'delhi NCR' : 'Delhi',
    'delhi ncr' : 'Delhi',

    'chenai' : 'chennai',
    'madras' : 'chennai'
}

dfc['customer_city'] = dfc['customer_city'].replace(city_map)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [12]:
import numpy as np

bool_cols = ['is_prime_member', 'is_prime_eligible', 'is_festival_sale']

for col in bool_cols:
    dfc[col] = dfc[col].replace(['', ' ', 'NA', 'N/A', None, 'None'], np.nan)


bool_map = {
    True: True,
    'True': True,
    'true': True,
    'Yes': True,
    'yes': True,
    'Y': True,
    'y': True,
     1: True,
    
     False: False,
    'False': False,
    'false': False,
    'No': False,
    'no': False,
    'N': False,
    'n': False,
     0: False
}


for col in bool_cols:
    dfc[col] = dfc[col].map(bool_map)

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.

In [13]:
dfc.columns = dfc.columns.str.strip()

category_map = {
    'electronics': 'Electronics',
    'ELECTRONICS': 'Electronics',
    'electronics & accessories': 'Electronics',
    'Electronicss': 'Electronics',
    'Electronics & Accessories': 'Electronics',
    'Electronic': 'Electronics',
    'clothing': 'Fashion'
}

dfc['category'] = dfc['category'].replace(category_map)

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [14]:
import numpy as np

days_map = {
    'Express': '0',
    'Same Day': '0',
    '-1': 'None',
    '1-2 days': '2'
}

dfc['delivery_days'] = dfc['delivery_days'].replace(days_map)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [15]:
dup_cols = [
    "customer_id",
    "product_id",
    "order_date",
    "final_amount_inr"
]

price_cols = [
    "original_price_inr",
    "discounted_price_inr",
    "subtotal_inr",
    "final_amount_inr"
]

dfc["dup_count"] = (
    dfc.groupby(dup_cols)["transaction_id"]
      .transform("count")
)


dfc["price_identical"] = (
    dfc.groupby(dup_cols)[price_cols]
      .transform("nunique")
      .max(axis=1) == 1
)


dfc["is_high_value"] = dfc["final_amount_inr"] > 5000
dfc["is_bulk_customer"] = dfc["customer_spending_tier"].isin(["Premium"])
dfc["is_bulk_quantity"] = dfc["quantity"] > 1

dfc["is_duplicate_candidate"] = dfc["dup_count"] > 1


In [16]:
df_deduped = dfc[dfc["is_duplicate_candidate"]].drop_duplicates(subset=dup_cols, keep="first")


In [17]:
print("Rows deleted:", (df_deduped))

Rows deleted:            transaction_id  order_date         customer_id   product_id  \
251     TXN_2021_00000252  2021-01-31  CUST_2017_00014282  PROD_000487   
318     TXN_2021_00000319  2021-01-24  CUST_2021_00000027  PROD_000148   
1519    TXN_2021_00001520  2021-01-07  CUST_2021_00006407  PROD_001891   
1651    TXN_2021_00001652  2021-01-03  CUST_2017_00011764  PROD_000579   
1835    TXN_2021_00001836  2021-01-10  CUST_2021_00030116  PROD_000263   
...                   ...         ...                 ...          ...   
136373  TXN_2021_00136374  2021-12-02  CUST_2021_00019617  PROD_000112   
136457  TXN_2021_00136458  2021-12-18  CUST_2021_00043621  PROD_000393   
136724  TXN_2021_00136725  2021-12-22  CUST_2015_00001755  PROD_000553   
137078  TXN_2021_00137079  2021-12-31  CUST_2021_00007145  PROD_000574   
137427  TXN_2021_00137428  2021-12-22  CUST_2019_00032092  PROD_000841   

                              product_name     category  subcategory    brand  \
251             

In [56]:
len(dfc)

77385

In [18]:
cols_to_drop = [
    "dup_count",
    "price_identical",
    "is_high_value",
    "is_bulk_customer",
    "is_bulk_quantity",
    "is_duplicate_candidate"
]

dfc = dfc.drop(columns=cols_to_drop)

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [20]:
import numpy as np

dfc["product_median_price"] = (
    dfc.groupby("product_id")["final_amount_inr"]
      .transform("median")
)

dfc["price_outlier"] = (
    dfc["final_amount_inr"] > 50 * dfc["product_median_price"]
)

dfc.loc[dfc["price_outlier"], "final_amount_inr"] /= 100
dfc.loc[dfc["price_outlier"], "discounted_price_inr"] /= 100
dfc.loc[dfc["price_outlier"], "original_price_inr"] /= 100

dfc["subtotal_inr"] = dfc["discounted_price_inr"] * dfc["quantity"]
dfc["final_amount_inr"] = dfc["subtotal_inr"] + dfc["delivery_charges"].fillna(0)

dfc["price_corrected_flag"] = dfc["price_outlier"]

dfc.drop(columns=["product_median_price"], inplace=True)

corrected_rows = dfc[dfc["price_corrected_flag"]]

print(corrected_rows)


Empty DataFrame
Columns: [transaction_id, order_date, customer_id, product_id, product_name, category, subcategory, brand, original_price_inr, discount_percent, discounted_price_inr, quantity, subtotal_inr, delivery_charges, final_amount_inr, customer_city, customer_state, customer_tier, customer_spending_tier, customer_age_group, payment_method, delivery_days, delivery_type, is_prime_member, is_festival_sale, festival_name, customer_rating, return_status, order_month, order_year, order_quarter, product_weight_kg, is_prime_eligible, product_rating, price_outlier, price_corrected_flag]
Index: []

[0 rows x 36 columns]


In [21]:
dfc = dfc.drop(
    columns=[
        "price_outlier",
        "price_corrected_flag"       
    ]
)



Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [23]:
dfc["payment_method"] = (
    dfc["payment_method"]
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.strip()
)

payment_map = {
    "UPI": "UPI",
    "PHONEPE": "UPI",
    "GOOGLEPAY": "UPI",
    "GPAY": "UPI",
    "PAYTM": "UPI",

    "CREDIT CARD": "Credit Card",
    "CREDIT_CARD": "Credit Card",
    "CC": "Credit Card",

    "DEBIT CARD": "Debit Card",
    "DC": "Debit Card",

    "COD": "Cash on Delivery",
    "CASH ON DELIVERY": "Cash on Delivery",

    "NET BANKING": "Net Banking"
}

dfc["payment_method"] = dfc["payment_method"].replace(payment_map)

In [24]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
    .astype(str)
    .str.replace('-', '', regex=False)
    .astype(float)
)


In [25]:
dfc["payment_method"].unique()

array(['UPI', 'Cash on Delivery', 'Credit Card', 'WALLET', 'Net Banking',
       'Debit Card'], dtype=object)

In [28]:
import pandas as pd

columns = [
    "transaction_id","order_date","customer_id","product_id","product_name",
    "category","subcategory","brand","original_price_inr","discount_percent",
    "discounted_price_inr","quantity","subtotal_inr"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

transaction_id: ['TXN_2021_00000001', 'TXN_2021_00000002', 'TXN_2021_00000003', 'TXN_2021_00000004', 'TXN_2021_00000005', 'TXN_2021_00000006', 'TXN_2021_00000007', 'TXN_2021_00000008', 'TXN_2021_00000009', 'TXN_2021_00000010', 'TXN_2021_00000011', 'TXN_2021_00000012', 'TXN_2021_00000013', 'TXN_2021_00000014', 'TXN_2021_00000015', 'TXN_2021_00000016', 'TXN_2021_00000017', 'TXN_2021_00000018', 'TXN_2021_00000019', 'TXN_2021_00000020', 'TXN_2021_00000021', 'TXN_2021_00000022', 'TXN_2021_00000023', 'TXN_2021_00000024', 'TXN_2021_00000025', 'TXN_2021_00000026', 'TXN_2021_00000027', 'TXN_2021_00000028', 'TXN_2021_00000029', 'TXN_2021_00000030', 'TXN_2021_00000031', 'TXN_2021_00000032', 'TXN_2021_00000033', 'TXN_2021_00000034', 'TXN_2021_00000035', 'TXN_2021_00000036', 'TXN_2021_00000037', 'TXN_2021_00000038', 'TXN_2021_00000039', 'TXN_2021_00000040', 'TXN_2021_00000041', 'TXN_2021_00000042', 'TXN_2021_00000043', 'TXN_2021_00000044', 'TXN_2021_00000045', 'TXN_2

In [26]:
import pandas as pd

columns = [
    "delivery_charges",
    "final_amount_inr","customer_city","customer_state","customer_tier",
    "customer_spending_tier","customer_age_group","payment_method","delivery_days",
    "delivery_type","is_prime_member","is_festival_sale",
    "festival_name"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

delivery_charges: ['0.0', '40.0']

final_amount_inr: ['10000.7', '100008.66', '100008.93', '100014.44', '100017.68', '100019.07', '10002.36', '100022.83', '100022.96', '10003.27', '100031.16', '100033.56', '100034.93', '10004.03', '100042.2', '100044.91', '100047.36000000002', '100048.72', '100048.98', '10005.02', '10005.04', '10005.67', '10005.68', '100056.93', '100059.42', '10006.1', '100062.48', '100063.08', '100066.5', '100068.24', '100073.14', '100073.68', '100076.58', '100077.07', '100078.24', '100081.52', '100083.73', '100083.97', '100086.31', '100088.67', '10009.56', '100102.41', '100106.55', '10012.21', '100120.75', '100124.12', '100127.28', '100128.12', '100128.51', '100134.54000000001', '100134.66', '10014.34', '100140.27', '100141.54', '100144.3', '100153.93', '100156.3', '100157.64', '100163.13', '100164.51', '100165.62', '100168.3', '10017.32', '100173.8', '100176.5', '100179.75', '100183.23000000001', '100183.53', '100186.52', '100189.29',

In [27]:
import pandas as pd

columns = [    "customer_rating","return_status","order_month","order_year","order_quarter",
    "product_weight_kg","is_prime_eligible","product_rating"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

customer_rating: ['3.0', '3.5', '4.0', '4.5', '5.0']

return_status: ['Cancelled', 'Delivered', 'Returned']

order_month: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']

order_year: ['2021']

order_quarter: ['1', '2', '3', '4']

product_weight_kg: ['0.03', '0.04', '0.05', '0.06', '0.07', '0.08', '0.09', '0.1', '0.11', '0.12', '0.13', '0.14', '0.15', '0.16', '0.17', '0.18', '0.19', '0.2', '0.21', '0.22', '0.23', '0.24', '0.25', '0.27', '0.28', '0.29', '0.3', '0.31', '0.32', '0.33', '0.34', '0.35', '0.36', '0.37', '0.38', '0.39', '0.4', '0.41', '0.42', '0.43', '0.44', '0.45', '0.46', '0.47', '0.48', '0.49', '0.5', '0.51', '0.52', '0.53', '0.54', '0.55', '0.56', '0.57', '0.58', '0.59', '0.61', '0.62', '0.63', '0.64', '0.65', '0.66', '0.67', '0.68', '0.69', '0.7', '0.71', '0.72', '0.73', '0.75', '0.78', '1.2', '1.21', '1.24', '1.29', '1.31', '1.33', '1.37', '1.39', '1.4', '1.46', '1.48', '1.5', '1.51', '1.55', '1.57', '1.58', '1.62', '1.63',

In [29]:
print(len(dfc.columns))
print(dfc.columns.tolist())


34
['transaction_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'original_price_inr', 'discount_percent', 'discounted_price_inr', 'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr', 'customer_city', 'customer_state', 'customer_tier', 'customer_spending_tier', 'customer_age_group', 'payment_method', 'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name', 'customer_rating', 'return_status', 'order_month', 'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible', 'product_rating']


In [30]:
dfc[dfc.duplicated()]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating


In [31]:
import pandas as pd

decimal_cols = dfc.select_dtypes(include=['float', 'float64']).columns

dfc[decimal_cols] = dfc[decimal_cols].round(2)
print(decimal_cols)


Index(['original_price_inr', 'discount_percent', 'discounted_price_inr',
       'subtotal_inr', 'delivery_charges', 'final_amount_inr',
       'customer_rating', 'product_weight_kg', 'product_rating'],
      dtype='object')


In [33]:
dfc.to_csv(r"C:\Users\hp\Desktop\Amazon\CSV_Clean_Files\amazon_india_2021_clean.csv",header='infer',index=False)

In [32]:
dfc

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2021_00000001,2021-01-29,CUST_2021_00014071,PROD_000178,Xiaomi Mi 5 64GB Black,Electronics,Smartphones,Xiaomi,43292.78,22.96,...,False,NaN,4.5,Delivered,1,2021,1,0.24,True,3.7
1,TXN_2021_00000002,2021-01-10,CUST_2019_00006495,PROD_000610,Vivo V15 Pro 256GB White,Electronics,Smartphones,Vivo,32964.72,16.68,...,False,NaN,5.0,Delivered,1,2021,1,0.19,True,4.1
2,TXN_2021_00000003,2021-01-18,CUST_2018_00029239,PROD_000404,Xiaomi Poco F1 128GB Gold,Electronics,Smartphones,Xiaomi,45794.75,0.00,...,False,NaN,NaN,Delivered,1,2021,1,0.22,True,4.4
3,TXN_2021_00000004,2021-01-08,CUST_2021_00040991,PROD_000664,Apple iPhone SE (2nd gen) 64GB Gold,Electronics,Smartphones,Apple,136337.08,0.00,...,False,NaN,4.0,Delivered,1,2021,1,0.17,True,3.8
4,TXN_2021_00000005,2021-01-02,CUST_2016_00006478,PROD_000466,Oppo A3s 64GB Black,Electronics,Smartphones,Oppo,19992.12,0.00,...,False,NaN,3.0,Delivered,1,2021,1,0.24,NaN,3.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138182,TXN_2021_00135877_DUP,NaN,CUST_2016_00000089,PROD_000402,Xiaomi Poco F1 128GB Blue,Electronics,Smartphones,Xiaomi,34242.15,23.22,...,False,NaN,5.0,Delivered,12,2021,4,0.19,False,3.4
138183,TXN_2021_00065427_DUP,2021-07-15,CUST_2017_00004814,PROD_001904,Xiaomi Tracker Premium,Electronics,Smart Watch,Xiaomi,60140.00,15.86,...,True,Prime Day,NaN,Delivered,7,2021,3,0.06,False,4.1
138184,TXN_2021_00012199_DUP,2021-02-01,CUST_2021_00012902,PROD_000904,Realme Realme Narzo 30 64GB Blue,Electronics,Smartphones,Realme,46453.23,28.15,...,NaN,NaN,4.0,Delivered,2,2021,1,0.19,False,3.8
138185,TXN_2021_00118907_DUP,2021-11-16,CUST_2021_00016509,PROD_001558,Lenovo VivoBook 4GB RAM Black,Electronics,Laptops,Lenovo,51021.72,0.00,...,False,NaN,3.0,Delivered,11,2021,4,2.76,True,4.0
